# İHA Video Arama — AU-AIR denemesi (Kaggle)

`poc/kaggle_pipeline_trial.ipynb`'nin sadeleştirilmiş hali — **sadece AU-AIR**
üzerinden gidiyoruz, SeaDroneSee/genel video akışı (kalibrasyon, toplu yükleme,
manifest.json karşılaştırması, tekne sorguları, vLLM) bu deftere alınmadı.

Amaç: "AU-AIR bizim kullanım amacımıza uyar mı" sorusunu **gerçek GPU'da**, 8
videonun TAMAMIYLA cevaplamak — `poc/auair_adapter.py` / `auair_build_videos.py`
/ `auair_ingest.py`'nin (production kodu değiştirilmeden, bkz. o dosyaların
docstring'leri) Kaggle'a taşınmış hali.

**Lisans:** AU-AIR CC BY-NC-SA/CC BY-NC (ticari/savunma kullanımına kapalı,
Qwen3-VL-Embedding-2B'nin VideoCLIP-XL/EBind'i elediği kararla aynı kategori)
— bu **sadece iç doğrulama** denemesi, kalıcı bir ingest yolu değil.

**SONUÇ VİDEOLAR GERÇEK ÇEKİM DEĞİL:** AU-AIR'in ayrık kareleri (~5 FPS, gerçek
zaman damgalarına sadık şekilde birleştirilmiş) kullanılıyor — bu MEKANİZMA
testi (ingest/pencereleme/embedding gerçekten çalışıyor mu), retrieval
KALİTESİ testi değil.

**Ayrı Qdrant koleksiyonu** (`clips_auair_test`) kullanılıyor.

Genel amaçlı (kendi videonuzu/SeaDroneSee'yi test etmek istiyorsanız) defter
için: [poc/kaggle_pipeline_trial.ipynb](kaggle_pipeline_trial.ipynb).

## Kullanım kuralları

- **Hücreleri sırayla çalıştırın.**
- **Ortam değişkenini `set_env(...)` ile değiştirin**, doğrudan `os.environ`
  ile değil (aşağıda tanımlı).
- **Qdrant istemcisini `close()` etmeyin** - önbellekli, gömülü mod dosya
  kilidi kullanıyor.
- Notebook'u **GitHub'dan taze açtığınızdan emin olun** - repo'yu güncellemek
  açık bir Kaggle sekmesindeki hücreleri otomatik güncellemez.
- Önce **Settings → Internet**'in açık olduğundan emin olun (Google Drive'dan
  indirme + git clone gerekiyor).

## 1. GPU + ortam kontrolü

**Önce:** sağ paneldeki **Settings → Accelerator**'dan bir GPU seçin.
**GPU T4 x2** önerilir (P100 değil) - T4'ün davranışını (bf16 yok, fp16'ya
geçiş) Colab'da doğruladık, P100 (Pascal, compute 6.0) farklı/daha eski bir
mimari ve AWQ kuantize modellerle sorunlu olabilir. "x2" yazsa da bu defter
tek GPU kullanıyor (çoklu-GPU paralelliği kurulmadı) - ikinci GPU boşta kalır.

GPU seçtikten sonra **Settings → Internet**'i açın ve oturumu yeniden
başlatın.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!free -g | head -2
!df -h /kaggle/working | tail -1

## 2. Depo + bağımlılıklar

Kaggle'da torch zaten CUDA'lı geldiği için `requirements.txt`'i olduğu gibi
kurmuyoruz (mevcut torch'u bozabilir). Sadece eksikleri kuruyoruz.

`git clone` başarısız olursa: yukarıdaki Internet ayarını kontrol edin.

In [ ]:
import pathlib, subprocess, sys

REPO = pathlib.Path('/kaggle/working/VideoAnalysis')

# Sessiz git komutlari kullanmiyoruz: pull sessizce basarisiz olursa ESKI
# KOD calismaya devam eder ve hata cok sonra alakasiz bir yerde patlar.
if REPO.exists():
    print('Depo mevcut, uzak surumle esitleniyor...')
    subprocess.run(['git', 'fetch', 'origin'], cwd=REPO, check=True)
    print(subprocess.run(['git', 'reset', '--hard', 'origin/main'], cwd=REPO,
                         capture_output=True, text=True).stdout.strip())
else:
    r = subprocess.run(['git', 'clone',
                        'https://github.com/ykyking1/VideoAnalysis.git', str(REPO)],
                       capture_output=True, text=True)
    print(r.stdout or r.stderr)

head = subprocess.run(['git', 'log', '--oneline', '-1'], cwd=REPO,
                      capture_output=True, text=True).stdout.strip()
print('\nCalisan surum:', head)

config_src = (REPO / 'common' / 'config.py').read_text(encoding='utf-8')
assert 'LOCAL_STORAGE_PATH' in config_src, (
    'ESKI KOD! git pull calismamis - yukaridaki "Calisan surum" satirini kontrol edin.')

%cd /kaggle/working/VideoAnalysis

# Kaggle'in CUDA'li torch'una DOKUNMUYORUZ - sadece eksik paketler.
# qwen-vl-utils>=0.0.14 kritik: eskisi SESSIZCE bozuk embedding uretiyor.
!pip install -q "transformers>=4.57" "qwen-vl-utils>=0.0.14" accelerate \
    qdrant-client ultralytics opencv-python-headless temporalio \
    pysolar shapely gdown 2>&1 | tail -3

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print('Kod guncel, hazir. Sonraki hucrede ortam degiskenleri ayarlanacak.')

## 3. Ortam yapılandırması

Kaggle'da Docker yok. İki servisi Docker'sız çalıştırıyoruz:

- **Qdrant** → gömülü mod (`QDRANT_LOCAL_PATH`)
- **Nesne deposu** → yerel dizin (`LOCAL_STORAGE_PATH`), MinIO'ya gerek yok

`QDRANT_COLLECTION` burada baştan `clips_auair_test` — bu defterde başka
koleksiyon kullanmıyoruz.

In [ ]:
import os, sys, importlib

def set_env(**kwargs):
    """Ortam degiskenini ayarlar VE common.config'i yeniden yukler.

    common/config.py env'i IMPORT ANINDA okuyor. Bir degiskeni sonradan
    degistirirseniz, config zaten yuklenmis oldugu icin surec-ici cagrilar
    ESKI degeri gorur. Bu yardimci o tuzagi kapatiyor - notebook boyunca
    env degistirmek icin hep bunu kullanin, dogrudan os.environ'a yazmayin."""
    for k, v in kwargs.items():
        os.environ[k] = str(v)
    if 'common.config' in sys.modules:
        importlib.reload(sys.modules['common.config'])

set_env(
    QDRANT_LOCAL_PATH='/kaggle/working/qdrant_data',
    LOCAL_STORAGE_PATH='/kaggle/working/storage',
    QDRANT_COLLECTION='clips_auair_test',        # AYRI koleksiyon - gercek/SeaDroneSee ile karismasin
    EMBEDDING_BATCH_SIZE=8,                      # T4 16GB icin baslangic
    EMBEDDING_DTYPE='auto',                      # T4 (compute 7.5) -> fp16
    CAPTION_ENABLED='false',                     # vLLM bu defterde yok
)

from common import config
from common.minio_client import backend_name
assert config.LOCAL_STORAGE_PATH, 'LOCAL_STORAGE_PATH okunmadi'
print('Nesne deposu :', backend_name())
print('Qdrant       : gomulu ->', config.QDRANT_LOCAL_PATH)
print('Koleksiyon   :', config.QDRANT_COLLECTION)
print('Batch        :', config.EMBEDDING_BATCH_SIZE)

!python -m scripts.init_storage --skip-postgres

## 4. Ortam doğrulaması

`torch ... CPU-only` ya da `qwen-vl-utils < 0.0.14` görürseniz **durun** —
ikisi de çökmeden sessizce bozuyor.

In [ ]:
!python -m scripts.check_env

## 5. AU-AIR verisini indir

Google Drive'dan iki dosya çekiliyor (idempotent - zaten indirilip açılmışsa
atlanır, oturum kesilip yeniden başlarsa baştan indirmez):

- annotations (~3.9MB, 32.823 kayıt)
- images (~2.2GB, 32.823 JPG)

In [ ]:
import pathlib, zipfile, json

AUAIR_DIR = pathlib.Path('/kaggle/working/auair_data')
AUAIR_DIR.mkdir(exist_ok=True)
ANNOT_ZIP = AUAIR_DIR / 'annotations.zip'
ANNOT_JSON = AUAIR_DIR / 'annotations.json'
IMAGES_ZIP = AUAIR_DIR / 'images.zip'
IMAGES_DIR = AUAIR_DIR / 'images'

if not ANNOT_JSON.exists():
    print('Annotations indiriliyor...')
    !python -m gdown 1boGF0L6olGe_Nu7rd1R8N7YmQErCb0xA -O {ANNOT_ZIP}
    with zipfile.ZipFile(ANNOT_ZIP) as z:
        z.extractall(AUAIR_DIR)
    # Zip icindeki gercek dosya adi degisebilir - annotations.json'a normalize et
    found = [p for p in AUAIR_DIR.rglob('*.json')]
    assert found, 'annotations.json zip icinde bulunamadi'
    if found[0] != ANNOT_JSON:
        found[0].rename(ANNOT_JSON)
    print('Annotations hazir:', ANNOT_JSON)
else:
    print('Annotations zaten mevcut, atlaniyor.')

if not IMAGES_DIR.exists() or not any(IMAGES_DIR.iterdir()):
    print('Goruntuler indiriliyor (2.2GB, biraz surer)...')
    !python -m gdown 1pJ3xfKtHiTdysX5G3dxqKTdGESOBYCxJ -O {IMAGES_ZIP}
    with zipfile.ZipFile(IMAGES_ZIP) as z:
        z.extractall(AUAIR_DIR)
    print('Goruntuler hazir:', IMAGES_DIR)
else:
    print('Goruntuler zaten mevcut, atlaniyor.')

n_annot = len(json.loads(ANNOT_JSON.read_text(encoding='utf-8'))['annotations'])
n_images = len(list(IMAGES_DIR.rglob('*.jpg')))
print(f'\n{n_annot} annotasyon, {n_images} goruntu dosyasi')
assert n_images >= n_annot, (
    f'Goruntu sayisi ({n_images}) annotasyon sayisindan ({n_annot}) az - indirme eksik olabilir.')

## 6. 8 videoyu inşa et

AU-AIR ayrık kareler (~5 FPS, bazı videolarda 15-52sn'lik gerçek boşluklar)
olarak geliyor - ffmpeg concat demuxer ile, kareler arası GERÇEK zaman
damgasına sadık kalarak birleştiriliyor (bkz. `poc/auair_build_videos.py`
docstring'i - sabit FPS varsayımı bu boşlukları sessizce yutardı).

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/VideoAnalysis')
from poc.auair_adapter import load_auair_records, split_by_source_video
from poc.auair_build_videos import build_video

records = load_auair_records(str(ANNOT_JSON))
groups = split_by_source_video(records)
print(f'{len(groups)} kaynak video, {len(records)} toplam kayit')

AUAIR_VIDEOS_DIR = AUAIR_DIR / 'videos'
AUAIR_VIDEOS_DIR.mkdir(exist_ok=True)

for prefix, group in sorted(groups.items()):
    out_path = AUAIR_VIDEOS_DIR / f'{prefix}.mp4'
    if out_path.exists():
        print(f'{prefix}: zaten var, atlaniyor')
        continue
    print(f'{prefix}: {len(group)} kare -> {out_path.name} ...')
    build_video(prefix, group, AUAIR_DIR / 'images', out_path)
    size_mb = out_path.stat().st_size / 1024**2
    print(f'  tamam: {size_mb:.1f} MB')

built = sorted(AUAIR_VIDEOS_DIR.glob('*.mp4'))
print(f'\n{len(built)}/8 video hazir: {[p.name for p in built]}')
assert len(built) == 8, 'Eksik video var - yukaridaki hatalari kontrol edin'

## 7. Embed + ingest (8 videonun tamamı)

Gerçek pipeline: proxy üretimi, AU-AIR telemetri adaptörü (MAVLink DEĞİL,
bkz. `poc/auair_adapter.py`), Qwen3-VL-Embedding-2B ile klip embedding,
YOLO26 görsel alanlar, Qdrant'a yazım (`clips_auair_test`). Caption
atlanıyor (vLLM bu defterde yok).

Yerel makinede (zayıf GPU) tek video (156sn) 27 dakika sürmüştü (0.10x
gerçek-zaman) - 8 videonun tamamı (~2,13 saat) o hızla ~21 saat sürerdi,
bu yüzden Kaggle GPU'suna taşındı. Aşağıdaki hücrenin sonunda çıkan
**agrege gerçek-zaman katsayısı** proje-ozeti.md §8'in 40x varsayımıyla
karşılaştırılacak en değerli sayı.

In [ ]:
from poc.auair_ingest import ingest_one

auair_results = []
for prefix in sorted(groups):
    r = await ingest_one(prefix, groups[prefix], AUAIR_VIDEOS_DIR, skip_caption=True)
    auair_results.append(r)

print(f"\n{'='*70}\nAU-AIR OZET ({len(auair_results)}/8 video)\n{'='*70}")
print(f"{'video':30}{'pencere':>9}{'sure(s)':>10}{'gercek-zaman':>14}{'arac':>7}")
total_dur = total_elapsed = 0.0
for r in auair_results:
    rt_factor = r['duration_s'] / r['elapsed_s']
    total_dur += r['duration_s']
    total_elapsed += r['elapsed_s']
    print(f"{r['video_id']:30}{r['windows']:>9}{r['duration_s']:>10.1f}"
          f"{rt_factor:>13.2f}x{r['vehicle_total']:>7}")

agg_factor = total_dur / total_elapsed
print(f"\nToplam: {total_dur:.1f}s video, {total_elapsed:.1f}s isleme suresi")
print(f"AGREGE gercek-zaman katsayisi: {agg_factor:.3f}x")
print(f"proje-ozeti.md §8 varsayimi: 40x  |  fark: {40/agg_factor:.0f}x daha yavas"
      if agg_factor > 0 else "")

## 8. Sorgu testleri

`clips_auair_test` üzerinde gerçek sorgular. vLLM yok - yapısal ayrıştırma
devre dışı, sorgu tamamen semantiğe düşüyor (`vehicle_count` filtresi manuel
kurulmadan test edilemez, bu defterde o yol yok - genel notebook'ta var).

In [ ]:
from query.pipeline import run_query
from scripts.query_cli import render

assert config.QDRANT_COLLECTION == 'clips_auair_test', (
    'Yanlis koleksiyon - 3. bolumdeki set_env calisti mi?')

AUAIR_PROMPTS = [
    'a truck at a roundabout',
    'an ambulance on the road',
    'cars waiting at an intersection',
    'a busy road with many vehicles',
    'a motorbike on the street',
]

for p in AUAIR_PROMPTS:
    print('=' * 70)
    print('SORGU:', p)
    render(run_query(p, top_k=5))
    print()

print('=' * 70)
print('BAKILACAK:')
print('  - Sonuclar mantikli mi (kavsak/trafik sahneleri donuyor mu)?')
print('  - video_id alani "auair_frame_..." formatinda mi (dogru koleksiyon)?')
print('  - agl_m/avg_speed_kmh degerleri fiziksel olarak makul mu '
      '(bkz. auair_adapter.py doc: ~5-30m, ~0-16 km/h)?')

## 9. Telemetri sanity kontrolü

`poc/auair_adapter.py`'nin docstring'inde belgelenen fiziksel-makullük
varsayımı: `agl_m` ~4.8-30.2m, `avg_speed_kmh` ~0.05-16 km/h (bkz. o
dosyanın DOĞRULANMAMIŞ VARSAYIMLAR bölümü - altitude mm, linear_x/y/z m/s
kabul edildi). Bu hücre 8 videonun TAMAMINDA gerçekten yazılan Qdrant
payload'larını tarayıp bu aralıkla karşılaştırıyor - `render()` bu
alanları basmadığı için 8. bölümdeki sorgu çıktısında görünmüyorlardı.

In [ ]:
from collections import defaultdict
from common.qdrant_store import get_client

assert config.QDRANT_COLLECTION == 'clips_auair_test', (
    'Yanlis koleksiyon - 3. bolumdeki set_env calisti mi?')

client = get_client()
points, offset = client.scroll(collection_name=config.QDRANT_COLLECTION, limit=1000,
                                with_payload=True, with_vectors=False)
while offset is not None:
    more, offset = client.scroll(collection_name=config.QDRANT_COLLECTION, limit=1000,
                                  offset=offset, with_payload=True, with_vectors=False)
    points += more

print(f'{len(points)} pencere taraniyor...\n')

per_video = defaultdict(list)
for p in points:
    pl = p.payload or {}
    per_video[pl.get('video_id')].append(pl)

# auair_adapter.py doc'undaki beklenen aralik - DOGRULANMAMIS varsayima
# dayanan bir REFERANS, kesin dogru kabul edilmemeli.
EXPECT_AGL = (4.8, 30.2)
EXPECT_SPEED = (0.05, 16.0)

print(f"{'video':30}{'agl_m (min-max)':>22}{'avg_speed_kmh (min-max)':>26}")
all_agl, all_speed = [], []
anomali = []
for vid in sorted(per_video):
    rows = per_video[vid]
    agl = [r['agl_m'] for r in rows if r.get('agl_m') is not None]
    speed = [r['avg_speed_kmh'] for r in rows if r.get('avg_speed_kmh') is not None]
    all_agl += agl
    all_speed += speed
    agl_s = f"{min(agl):.1f}-{max(agl):.1f}" if agl else "(yok)"
    speed_s = f"{min(speed):.2f}-{max(speed):.2f}" if speed else "(yok)"
    print(f"{vid:30}{agl_s:>22}{speed_s:>26}")
    if agl and (min(agl) < EXPECT_AGL[0] * 0.5 or max(agl) > EXPECT_AGL[1] * 1.5):
        anomali.append(f"{vid}: agl_m araligi disinda ({agl_s}, beklenen ~{EXPECT_AGL})")
    if speed and max(speed) > EXPECT_SPEED[1] * 1.5:
        anomali.append(f"{vid}: avg_speed_kmh beklenenden fazla ({speed_s}, beklenen ~{EXPECT_SPEED})")

print(f"\nTUMU: agl_m {min(all_agl):.1f}-{max(all_agl):.1f}m  |  "
      f"avg_speed_kmh {min(all_speed):.2f}-{max(all_speed):.2f} km/h")
print(f"Referans (auair_adapter.py doc): agl_m ~{EXPECT_AGL}, avg_speed_kmh ~{EXPECT_SPEED}")

if anomali:
    print(f"\n!! {len(anomali)} anomali:")
    for a in anomali:
        print(' -', a)
else:
    print("\nTum videolar referans araligina yakin - birim varsayimi (mm/m-s-1) tutarli gorunuyor.")

## 10. vLLM — yapısal ayrıştırma (isteğe bağlı, ağır)

**Bu hat hiç doğrulanmadı** — projenin en büyük doğrulanmamış parçası
(bkz. `kaggle_pipeline_trial.ipynb`'nin 10. bölümü - aynı kurulum, burada
AU-AIR verisine karşı deneniyor).

T4 16GB'de embedding modeli (~4,3 GB) + 7B-AWQ (~5 GB) birlikte sığar ama
sıkışıktır - önce embedding modelini boşaltıyoruz.

In [ ]:
from ingest.activities.clip_embedding import unload_model
unload_model()

In [ ]:
import subprocess

# SURUM GECMISI (bkz. kaggle_pipeline_trial.ipynb 10. bolum) - GUNCEL vLLM'i
# ihtiyaci olan torch surumuyle birlikte ACIKCA istiyoruz.
!pip install -q uv
!uv pip install -q --system "torch==2.11.0" vllm xgrammar --torch-backend=auto 2>&1 | tail -20

# DOGRULAMA - tahmin etmiyoruz, GORUYORUZ: torch GERCEKTEN 2.11.0 mu?
result = subprocess.run(['python3', '-c',
    'import torch, vllm; print(f"TORCH: {torch.__version__}"); print(f"VLLM: {vllm.__version__}")'],
    capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('import edilemedi:', result.stderr[-500:])
elif '2.11.0' not in result.stdout.split('VLLM')[0]:
    print('*** UYARI: torch 2.11.0 DEGIL - sabitleme etkisiz kaldi. ***')
    print('Cekirdegi yeniden baslatip bu hucreyi tekrar calistirin.')

In [ ]:
import time
from common.llm import health_check

LOG = '/kaggle/working/vllm.log'
vllm_log = open(LOG, 'w')
import subprocess as sp
vllm_proc = sp.Popen([
    'vllm', 'serve', 'Qwen/Qwen2.5-7B-Instruct-AWQ',
    '--structured-outputs-config.backend', 'xgrammar',
    '--gpu-memory-utilization', '0.45',
    '--max-model-len', '2048',
    '--dtype', 'half',   # T4 (compute 7.5) bfloat16 desteklemiyor, half=fp16
], stdout=vllm_log, stderr=sp.STDOUT)
print(f'vLLM baslatiliyor (pid={vllm_proc.pid}) - birkac dakika surer')
print(f'Log: {LOG}   ->  !tail -30 {LOG}')

for i in range(60):
    if vllm_proc.poll() is not None:
        log_tail = open(LOG, encoding='utf-8', errors='replace').read()[-3000:]
        print(f'\nvLLM SUREC OLDU (cikis kodu {vllm_proc.returncode}). Log sonu:')
        print(log_tail)
        break
    if health_check():
        print(f'vLLM hazir ({i*10}s)')
        break
    time.sleep(10)
else:
    print('vLLM 10 dakikada acilmadi. Log sonu:')
    !tail -25 {LOG}

In [ ]:
from query.pipeline import run_query
from scripts.query_cli import render

assert config.QDRANT_COLLECTION == 'clips_auair_test', 'Yanlis koleksiyon'

AUAIR_STRUCTURAL_PROMPTS = [
    'en az 10 arac gorunen kayitlar',
    '20 metreden alcakta ucan',
    '1000 arac olan goruntuler',        # imkansiz -> gevsetme tetiklenmeli
    'a truck at a roundabout',          # tamamen semantik -> filtre CIKMAMALI (beklenti)
]

for p in AUAIR_STRUCTURAL_PROMPTS:
    print('=' * 70)
    print('SORGU:', p)
    render(run_query(p))
    print()

print('=' * 70)
print('BAKILACAK:')
print('  1. "10 arac" -> min_vehicle_count=10 olmali')
print('  2. "20 metreden alcakta" -> max_agl_m=20 olmali')
print('  3. son sorgu (tamamen semantik) -> "filtre yok" olmali')
print('  4. "gecikme: parse=...ms" -> vLLM ayristirmanin GERCEK maliyeti')

## 11. YOLO vs AU-AIR gerçek etiketleri (araç sayımı doğruluğu)

Şu ana kadarki `vehicle_count` alanı tamamen **bizim** YOLO26'mızın (COCO
ön-eğitimli, aerial için fine-tune EDİLMEDİ) tahminiydi - AU-AIR'in kendi
gerçek `bbox` etiketleriyle hiç karşılaştırılmamıştı. AU-AIR'in kendi
etiketleri var (`Human, Car, Truck, Van, Motorbike, Bicycle, Bus, Trailer`)
- bu hücre onları gerçek referans olarak kullanıyor
(SeaDroneSee `manifest.json` karşılaştırmasıyla aynı mantık,
`kaggle_pipeline_trial.ipynb`'nin 8. bölümü).

**Metodoloji notları:**
- Bizim `count_vehicles()` (`ingest/activities/visual_fields.py`) şu COCO
  sınıflarını "araç" sayıyor: `boat, car, truck, bus, train, airplane,
  motorcycle`. AU-AIR'in `Van`, `Bicycle`, `Trailer` sınıflarının COCO
  karşılığı yok - **adil karşılaştırma için GT'den de dışlandı**, sadece
  `Car/Truck/Motorbike/Bus` sayılıyor. `Human` zaten iki tarafta da yok.
- Pencere içi karşılaştırma **max eş-zamanlı** mantığıyla (bizim
  `count_vehicles()` ile aynı yöntem): pencereye düşen AU-AIR karelerinin
  her birinde GT sayımı yapılıp maksimum alınıyor, toplam değil.
- Pencere sınırında (`t_start`/`t_end` tam eşleşen kare) küçük bir çakışma
  riski var (`<=` kullanıldı) - ihmal edilebilir, N küçük.

In [ ]:
import json
from collections import defaultdict
from common.qdrant_store import get_client

AUAIR_CATEGORIES = ['Human', 'Car', 'Truck', 'Van', 'Motorbike', 'Bicycle', 'Bus', 'Trailer']
# YOLO26'nin VEHICLE_LIKE_CLASSES kumesiyle (ingest/activities/visual_fields.py)
# esleyen AU-AIR siniflari. Van/Bicycle/Trailer'in COCO karsiligi yok - GT'DEN
# DE DISLANDI (yontem tutarli: iki taraf da ayni tanimi kullansin diye).
GT_VEHICLE_CLASSES = {'Car', 'Truck', 'Motorbike', 'Bus'}
GT_VEHICLE_IDX = {AUAIR_CATEGORIES.index(c) for c in GT_VEHICLE_CLASSES}
print('GT arac siniflari:', GT_VEHICLE_CLASSES, '(indeksler:', GT_VEHICLE_IDX, ')')
print('DISLANAN (COCO karsiligi yok): Human, Van, Bicycle, Trailer\n')

raw = json.loads(ANNOT_JSON.read_text(encoding='utf-8'))
assert raw['categories'] == AUAIR_CATEGORIES, 'Kategori sirasi beklenenden farkli!'
gt_count_by_image = {}
for ann in raw['annotations']:
    gt_count_by_image[ann['image_name']] = sum(
        1 for b in ann['bbox'] if b['class'] in GT_VEHICLE_IDX)

assert config.QDRANT_COLLECTION == 'clips_auair_test', 'Yanlis koleksiyon'
client = get_client()
points, offset = client.scroll(collection_name=config.QDRANT_COLLECTION, limit=1000,
                                with_payload=True, with_vectors=False)
while offset is not None:
    more, offset = client.scroll(collection_name=config.QDRANT_COLLECTION, limit=1000,
                                  offset=offset, with_payload=True, with_vectors=False)
    points += more

per_video = defaultdict(list)
for p in points:
    pl = p.payload or {}
    per_video[pl.get('video_id')].append(pl)

print(f"{'video':30}{'pencere':>8}{'MAE':>8}{'birebir dogru':>16}")
all_errors = []
for video_id in sorted(per_video):
    prefix = video_id.replace('auair_', '', 1)
    group = groups[prefix]   # 6. bolumde hesaplandi - hala bellekte
    errors = []
    for w in sorted(per_video[video_id], key=lambda r: r['t_start']):
        frames_in_window = [r for r in group if w['t_start'] <= r['t'] <= w['t_end']]
        gt = max((gt_count_by_image.get(r['image_name'], 0) for r in frames_in_window), default=0)
        yolo = w.get('vehicle_count', 0) or 0
        errors.append(yolo - gt)
    all_errors += errors
    mae = sum(abs(e) for e in errors) / len(errors) if errors else float('nan')
    exact = sum(1 for e in errors if e == 0)
    print(f"{video_id:30}{len(errors):>8}{mae:>8.2f}{f'{exact}/{len(errors)}':>16}")

overall_mae = sum(abs(e) for e in all_errors) / len(all_errors)
overall_exact = sum(1 for e in all_errors if e == 0)
print(f"\nTUMU: N={len(all_errors)}  MAE={overall_mae:.2f}  "
      f"birebir dogru={overall_exact}/{len(all_errors)} "
      f"({100*overall_exact/len(all_errors):.0f}%)")
print(f"Ortalama fark (YOLO-GT, isaretli): {sum(all_errors)/len(all_errors):+.2f}  "
      f"(pozitif = YOLO fazla sayiyor, negatif = eksik sayiyor)")

## 12. `vehicle_count`'u AU-AIR gerçek etiketiyle düzelt

11. bölüm YOLO'nun (COCO ön-eğitimli, aerial fine-tune YOK) bu veri
setinde `vehicle_count`'u sistematik olarak eksik saydığını gösterdi
(recall %16 - bkz. worklog). **YOLO'nun asıl amacı, insan etiketi
OLMAYAN gerçek arşiv videolarında bu alanı üretmek** - AU-AIR'de zaten
gerçek etiket var, bu yüzden bundan sonraki sorgu denemelerinin (8. ve
10. bölümdeki gibi) güvenilir sayılarla çalışması için `clips_auair_test`
koleksiyonundaki `vehicle_count`'u AU-AIR'in kendi etiketiyle **yerinde
değiştiriyoruz** (`set_payload` - embedding/telemetriye dokunulmuyor,
ucuz bir işlem).

**Not:** bu SADECE izole test koleksiyonunda yapılıyor - gerçek `clips`
koleksiyonuna dokunulmuyor, orada zaten insan etiketi yok ve YOLO'nun
kendisi hâlâ tek kaynak. `vehicle_count_source` payload alanı eklenip
bu koleksiyonun artık ground-truth kaynaklı olduğu işaretleniyor.

In [ ]:
import json
from common.qdrant_store import get_client

AUAIR_CATEGORIES = ['Human', 'Car', 'Truck', 'Van', 'Motorbike', 'Bicycle', 'Bus', 'Trailer']
GT_VEHICLE_CLASSES = {'Car', 'Truck', 'Motorbike', 'Bus'}
GT_VEHICLE_IDX = {AUAIR_CATEGORIES.index(c) for c in GT_VEHICLE_CLASSES}

raw = json.loads(ANNOT_JSON.read_text(encoding='utf-8'))
gt_count_by_image = {}
for ann in raw['annotations']:
    gt_count_by_image[ann['image_name']] = sum(
        1 for b in ann['bbox'] if b['class'] in GT_VEHICLE_IDX)

assert config.QDRANT_COLLECTION == 'clips_auair_test', 'Yanlis koleksiyon'
client = get_client()
points, offset = client.scroll(collection_name=config.QDRANT_COLLECTION, limit=1000,
                                with_payload=True, with_vectors=False)
while offset is not None:
    more, offset = client.scroll(collection_name=config.QDRANT_COLLECTION, limit=1000,
                                  offset=offset, with_payload=True, with_vectors=False)
    points += more

updated = 0
for p in points:
    pl = p.payload or {}
    video_id = pl.get('video_id')
    prefix = video_id.replace('auair_', '', 1)
    group = groups[prefix]   # 6. bolumde hesaplandi - hala bellekte
    frames_in_window = [r for r in group if pl['t_start'] <= r['t'] <= pl['t_end']]
    gt = max((gt_count_by_image.get(r['image_name'], 0) for r in frames_in_window), default=0)
    if gt != pl.get('vehicle_count'):
        client.set_payload(
            collection_name=config.QDRANT_COLLECTION,
            payload={'vehicle_count': gt, 'vehicle_count_source': 'auair_ground_truth'},
            points=[p.id],
        )
        updated += 1
    else:
        client.set_payload(
            collection_name=config.QDRANT_COLLECTION,
            payload={'vehicle_count_source': 'auair_ground_truth'},
            points=[p.id],
        )

print(f'{updated}/{len(points)} pencerenin vehicle_count degeri AU-AIR gercek etiketiyle guncellendi.')
print('Tum pencerelere vehicle_count_source=auair_ground_truth payload alani eklendi.')
print('Bundan sonraki sorgu/filtre denemeleri artik YOLO degil, gercek etiketle calisiyor.')

## 13. Sonuçları kaydedin

Bu denemeden çıkan, proje-ozeti.md §8'e ve dataset kararına girecek sayılar:

1. **Agrege gerçek-zaman katsayısı** (7. bölümün özet tablosu) — yerel
   makinede ölçülen 0.10x ile karşılaştırın; T4'te ne kadar farklı çıkıyor,
   §8'in 40x varsayımına ne kadar yaklaşıyor.
2. **8 videonun tamamı gerçekten ingest edildi mi** — hata alan video oldu
   mu, oldu ise hangi adımda (proxy/telemetri/embedding/YOLO/yazım).
3. **Sorgu sonuçları mantıklı mı** — "a truck at a roundabout" gibi
   sorgular gerçekten kavşak/trafik sahneleri mi döndürüyor, yoksa alakasız
   mı (retrieval KALİTESİ testi değil ama tamamen rastgele de olmamalı).
4. **Telemetri alanları fiziksel olarak makul mü** (9. bölüm) — 8 videonun
   tamamında `agl_m`/`avg_speed_kmh` aralıkları `poc/auair_adapter.py`'nin
   docstring'inde belirtilen aralıklarla (4.8-30.2m, 0.05-16 km/h) tutarlı
   mı, yoksa bazı videolarda sapma var mı.
5. **vLLM yapısal ayrıştırma doğruluğu** (10. bölüm) — alanlar doğru
   dolduruluyor mu, tamamen semantik sorgularda gereksiz filtre çıkıyor mu,
   `parse=...ms` gecikmesi ne kadar (fallback yolun ~15ms'lik maliyetiyle
   karşılaştırın).
6. **YOLO'nun araç sayımı doğruluğu** (11. bölüm) — AU-AIR'in kendi
   etiketlerine göre MAE ve birebir doğru oranı ne kadar, COCO ön-eğitimli
   (aerial için fine-tune EDİLMEDİ) modelin bu görev için ne kadar
   güvenilir olduğuna dair ilk somut sayı. **Ölçüldü: recall %16 (bkz.
   worklog) - hard filtrenin gevşetme olmadan ne kadar kayıp verdiğine
   dair ikinci, bağımsız bir kanıt.**
7. **12. bölümden sonra** `vehicle_count` artık AU-AIR gerçek etiketi -
   önceki bölümlerdeki (8, 10) YOLO-kaynaklı sonuçlarla bu bölümden
   sonrakileri karıştırmayın.

**Unutmayın:** bu deneme AU-AIR'in **CC BY-NC-SA/CC BY-NC** lisansı altında,
sadece iç doğrulama amaçlı. Sonuçlar "AU-AIR mekanizma testi olarak işe
yarıyor mu" sorusuna cevap veriyor — "hangi embedding modeli/pencereleme
daha iyi" sorusuna değil (bunun için golden set gerekiyor, §7 hâlâ boş).

Sonuçları bu notebook'un çıktısıyla birlikte paylaşın; `docs/` altına yeni
bir worklog girişi olarak işlenecek.